<a href="https://colab.research.google.com/github/muraiukiha/face-recognition/blob/main/Eigen_faces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
顔認識器インターフェース
参考:
- Python公式ドキュメント abc
  https://docs.python.org/ja/3/library/abc.html
"""
from abc import ABC, abstractmethod
import numpy as np

class FeatureExtractor(ABC):
    """特徴量抽出器インターフェース"""
    def learn(self, pictures: np.ndarray, label: np.ndarray | None = None) -> "FeatureExtractor":
        """抽出方法の学習(PCA専用)
        pictures: 登録画像 (枚数, 縦幅, 横幅)
        label: 各画像の人物ID (枚数,)。使わない手法では None
        """
        return self

    @abstractmethod
    def outputFeature(self, pictures: np.ndarray) -> np.ndarray:
        """顔画像を特徴量ベクトルに変換する
        pictures: 顔画像（登録画像またはテスト画像） (枚数, 縦幅, 横幅)
        戻り値: 特徴量ベクトル (枚数, 特徴量の次元)
        """

class Checker(ABC):
    """照合器インターフェース"""
    classes_: np.ndarray
    @abstractmethod
    def fit(self, features: np.ndarray, label: np.ndarray) -> "Checker":
        """登録画像の特徴量と、その人のID（ラベル)を受け取り、登録する。
        features: 登録画像の特徴量
        label: 登録者のID
        """
    @abstractmethod
    def score(self, testFeatures: np.ndarray) -> np.ndarray:
        """テスト画像の特徴量を受け取り、登録者全員との照合結果のスコアを返す
        testFeatures: テストデータの特徴量(枚数、次元数)
        戻り値: スコア（枚数、人数）。[i, j]はi枚目とその人のスコア（大きいほど似ているように！）
        """

    def predict(self, features: np.ndarray) -> np.ndarray:
        """各テスト画像について、スコアが最大の人IDを返す
        features: テストデータの特徴量
        """
        return self.classes_[self.score(features).argmax(axis=1)]

In [3]:
from sklearn.decomposition import PCA
class Eigenfaces(FeatureExtractor):

    def __init__(self, n_components=100):
        self.n_components = n_components

    def learn(self,pictures, label=None):
        X = np.reshape(pictures, (pictures.shape[0],-1))
        self.pca = PCA(n_components= self.n_components)
        self.pca.fit(X)
        return self
    def outputFeature(self, pictures):
        X = np.reshape(pictures,(pictures.shape[0],-1))
        return self.pca.transform(X)

class Saikinbo(Checker):
    def fit(self,features,label):
        self.features = features
        self.label = label
        self.classes_ = np.unique(label)
        return self
    def score(self, testFeatures):
        scores = np.zeros((len(testFeatures), len(self.classes_)))
        for test_number in range(len(testFeatures)):
          for i, id in enumerate(self.classes_):
            distances = []
            a = testFeatures[test_number] #(100,)
            person_features = self.features[self.label == id]
            for number in range(len(person_features)):
              b = person_features[number]
              distances.append(np.linalg.norm(a-b))
            scores[test_number,i]=-np.min(distances)#これが各一人の最大スコア
        return scores





